In [ ]:
import jax
from jax import config
config.update("jax_enable_x64", True)
import jax.numpy as jnp
import numpyro
numpyro.set_host_device_count(4)
import numpyro.distributions as dist
from numpyro.infer import MCMC, NUTS

import matplotlib.pyplot as plt

# Mesurmets taken from Particle Data Group on 20. December 2025
# mass of leptons = (electron, muon, tauon)
m_obs = jnp.array([0.51099895000, 105.6583755, 1776.930], dtype=jnp.float64)
sigma_m_obs = jnp.array([0.00000000015, 0.0000023, 0.09], dtype=jnp.float64)

def model(m_obs, sigma_m_obs):
    c_1 = numpyro.sample("c_1", dist.Normal(1.0, 0.2))
    c_2 = numpyro.sample("c_2", dist.Normal(1.0, 0.2))
    c_3 = numpyro.sample("c_3", dist.Normal(1.0, 0.2))

    eps = numpyro.sample("eps", dist.Normal(0.01, 0.03))

    v = 246000.0
    pref = v / jnp.sqrt(2.0)
    m_pred = pref * jnp.array([c_1 * eps**3, c_2 * eps**2, c_3 * eps])

    numpyro.sample("logm", dist.Normal(m_pred, sigma_m_obs), obs=m_obs)

key = jax.random.PRNGKey(0)

# NUTS kernel + MCMC runner
kernel = NUTS(model, target_accept_prob=0.995, dense_mass=True)
mcmc = MCMC(kernel, num_warmup=20000, num_samples=5000, num_chains=4, progress_bar=True)

# Run sampling
mcmc.run(key, m_obs=m_obs, sigma_m_obs=sigma_m_obs)

# Print a summary (means, std, ESS, r_hat, etc.)
mcmc.print_summary()

# Extract samples (dict of arrays)
samples = mcmc.get_samples(group_by_chain=False)


c:\Users\matic\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Compiling.. :   0%|          | 0/25000 [00:00<?, ?it/s]





Running chain 0:   0%|          | 0/25000 [00:02<?, ?it/s]













Running chain 2: 100%|██████████| 25000/25000 [00:04<00:00, 5637.45it/s] 




Running chain 0:   5%|▌         | 1250/25000 [00:08<01:56, 204.08it/s]



Running chain 0:  10%|█         | 2500/25000 [00:14<01:50, 203.93it/s]



Running chain 0:  15%|█▌        | 3750/25000 [00:21<01:45, 201.62it/s]



Running chain 0:  20%|██        | 5000/25000 [00:27<01:40, 198.28it/s]



Running chain 0:  25%|██▌       | 6250/25000 [00:33<01:34, 198.83it/s]



Running chain 0:  30%|███       | 7500/25000 [00:39<01:27, 199.84it/s]

Running chain 0:  35%|███▌      | 8750/25000 [00:46<01:21, 200.41it/s


                mean       std    median      5.0%     95.0%     n_eff     r_hat
       c_1     -0.29      0.76     -0.29     -1.32      0.75       nan 1278353874.71
       c_2     -0.21      0.91     -0.39     -1.09      1.03       nan 959700567.40
       c_3     -0.13      1.13     -0.16     -1.61      1.40      2.00 2446204007.01
       eps      0.34      0.60     -0.01     -0.02      1.38       nan 16689958154.37

Number of divergences: 5270
